In [13]:
import numpy as np
import pandas as pd
import json

np.bool = np.bool_
from docplex.cp.model import *

data_file_path = '../data/생관3-612-24-031_2025년 1월 일일생산계획 및 업체별 차종현황_Rev.00_24.12.31(작성중).xlsx'

In [14]:
# 공정 리스트
df_process = pd.read_excel(data_file_path, skiprows=[0, 1, 2], usecols='C:D', nrows=64)
df_vehicle = pd.read_excel(data_file_path, skiprows=[0, 1, 2, 3], usecols='F:BG', nrows=62)
df_process = df_process.drop(index=df_process.index[0])
df_process = df_process.reset_index(drop=True)
df_process = df_process.map(lambda x: x.strip() if isinstance(x, str) else x)
df_process = df_process.ffill()
df_vehicle.index = df_process['세부공정명'].tolist()
df_process.drop_duplicates(inplace=True)
df_process = df_process.reset_index(drop=True)

In [15]:
class Operation:
    def __init__(self, operation_name, order, test_operation, type_TC, department):
        self.operation_name = operation_name
        self.order = order
        self.test_operation = test_operation  # True or False
        self.type_TC = type_TC  # True or False
        self.department = department
        # self.position = position
        self.duration = 1
        self.resource = 1

In [16]:
process_list = list(df_process['세부공정명'])
operation_list = ["operation" + str(i) for i in range(len(process_list))]
test_list = [1, 8, 9, 10, 13, 19, 29, 33, 34, 36, 37, 42, 44, 47]
TC_list = [14, 20, 23, 26, 30, 32]
operation_dict = {"operation" + str(i):Operation(row['세부공정명'], i, i in test_list, i in TC_list, row['담당공정']) for i, row in df_process.iterrows()}    

In [17]:
same_sequence_constraint = [[1, 2], [38, 39, 40, 41], [42, 43], [46, 47]]

In [18]:
class Calendar:
    def __init__(self):
        self.holiday = [1, 5, 12, 19, 26, 28, 29, 30] 
        self.saturday = [4, 11, 18, 25]
        self.index_to_day_calendar_dict = {i: pd.to_datetime('2025-01-01') + pd.Timedelta(days=day - 1) for i, day in enumerate(d for d in range(1, 32) if d not in self.holiday)}
        self.day_to_index_calendar_dict = {value:key for key, value in self.index_to_day_calendar_dict.items()}

In [19]:
calendar = Calendar()

In [21]:
class Vehicle:
    def __init__(self, vehicle_name, type_TC):
        self.vehicle_name = vehicle_name
        self.type_TC = type_TC
        self.operation_dict = dict()
    
    def add_operation(self, operation_name, date):
        if operation_name not in self.operation_dict:
            self.operation_dict[operation_name] = date

In [22]:
columns = df_vehicle.columns.tolist()
for i in range(1, len(columns)):  # 첫 번째 열은 처리할 필요 없음
    if isinstance(columns[i], str) and "Unnamed" in columns[i]:
        columns[i] = columns[i - 1]  # 바로 왼쪽 열 이름으로 대체
df_vehicle.columns = columns

In [29]:
vehicle_dict = dict()
for row_idx in range(df_vehicle.shape[0]):
    operation_name = df_vehicle.index[row_idx]
    for col_idx in range(df_vehicle.shape[1]):
        date = df_vehicle.columns[col_idx]
        vehicle_name = df_vehicle.iloc[row_idx, col_idx]
        
        if pd.notna(vehicle_name):
            if vehicle_name not in vehicle_dict:
                vehicle_dict[vehicle_name] = Vehicle(vehicle_name, True if 'TC' in vehicle_name else False)
            vehicle_dict[vehicle_name].add_operation(operation_list[process_list.index(operation_name)], date)

In [42]:
vehicle_dict["S188-M1"].operation_dict

{'operation0': 3,
 'operation1': 6,
 'operation2': 6,
 'operation3': 21,
 'operation4': 22,
 'operation5': 23,
 'operation6': 24,
 'operation8': 27,
 'operation9': 24,
 'operation10': 27}